In [1]:
import pandas as pd

data_path= "../data/cleaned/"

# reading all the csv files
customers= pd.read_csv(data_path + "customers.csv", dtype={"customer_zip_code_prefix": str})
geolocation= pd.read_csv(data_path + "geolocation.csv",dtype={"geolocation_zip_code_prefix": str})
order_items= pd.read_csv(data_path + "order_items.csv")
order_payments= pd.read_csv(data_path + "order_payments.csv")
order_reviews= pd.read_csv(data_path + "order_reviews.csv")
orders= pd.read_csv(data_path + "orders.csv")
products= pd.read_csv(data_path + "products.csv")
sellers= pd.read_csv(data_path + "sellers.csv",dtype={"seller_zip_code_prefix": str})
category_translation= pd.read_csv(data_path + "category_name_translation.csv")

## SQL connection

In [2]:
# installing the required packages for mysql
%pip install sqlalchemy pymysql python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# connect to mysql
import os
import sqlalchemy as sqla
from dotenv import load_dotenv
from urllib.parse import quote_plus

# to load the .env file load_dotenv is used here
load_dotenv('../.env')

# all the credentials loaded from the .env file
username=os.getenv('DB_USER')
pwd=os.getenv('DB_PASSWORD')
host=os.getenv('DB_HOST')
port=os.getenv('DB_PORT')
db= os.getenv('DB_NAME')

# quote_plus encodes the password so that any special characters in password can't break url
safe_pwd=quote_plus(pwd)

# connection url
url='mysql+pymysql://'+username+':'+safe_pwd+'@'+host+':'+port

# creating the engine
con=sqla.create_engine(url)

with con.connect() as conn:
    print("Connected")
# so connection is wroking properly

Connected


#### Database Creation

In [4]:
# Database creation
with con.connect() as conn:
    conn.execute(sqla.text('drop database if exists cart2insights'))
    conn.execute(sqla.text('create database cart2insights'))
    print('DB created')

# creating database connection
db_con=sqla.create_engine(url+'/cart2insights')

with db_con.connect() as db_conn:
    print('connected to database')

DB created
connected to database


#### Tables creation

In [5]:
table_queries=[
""" create table category_translation 
(
    product_category_name varchar(100) primary key,
    product_category_name_english varchar(100)
)""",

""" create table geolocation
(
    geolocation_zip_code_prefix varchar(5) primary key,
    geolocation_lat double,
    geolocation_lng double
)""",

""" create table customers
(
    customer_id varchar(50) primary key,
    customer_unique_id varchar(50),
    customer_zip_code_prefix varchar(5),
    customer_city varchar(50),
    customer_state varchar(50)
)""",

""" create table sellers
(
    seller_id varchar(50) primary key,
    seller_zip_code_prefix varchar(5),
    seller_city varchar(50),
    seller_state varchar(50)
)""",

""" create table products
(
    product_id varchar(50) primary key,
    product_category_name varchar(100),
    product_name_length int,
    product_description_length int,
    product_photos_qty int,
    product_weight_g float,
    product_length_cm float,
    product_height_cm float,
    product_width_cm float,
    foreign key(product_category_name) references category_translation(product_category_name)
)""",

""" create table orders
(
    order_id varchar(50) primary key,
    customer_id varchar(50),
    order_status varchar(50),
    order_purchase_timestamp datetime,
    order_approved_at datetime,
    order_delivered_carrier_date datetime,
    order_delivered_customer_date datetime,
    order_estimated_delivery_date datetime,
    foreign key(customer_id) references customers(customer_id)
)""",

""" create table order_items
(
    order_id varchar(50),
    order_item_id int,
    product_id varchar(50),
    seller_id varchar(50),
    shipping_limit_date datetime,
    price decimal(10,2),
    freight_value decimal(10,2),
    primary key(order_id,order_item_id),
    foreign key(order_id) references orders(order_id),
    foreign key(product_id) references products(product_id),
    foreign key(seller_id) references sellers(seller_id)
 )""",

""" create table order_payments
(
    order_id varchar(50),
    payment_sequential int,
    payment_type varchar(50),
    payment_installments int,
    payment_value decimal(10,2),
    primary key(order_id,payment_sequential),
    foreign key(order_id) references orders(order_id)
)""",

""" create table order_reviews
(
    review_id varchar(50),
    order_id varchar(50),
    review_score int,
    review_comment_title varchar(300),
    review_comment_message text,
    review_creation_date datetime,
    review_answer_timestamp datetime,
    primary key(review_id,order_id),
    foreign key(order_id) references orders(order_id)
)
"""
]
# once all the tables are added to table_queries list, use this to create tables in the database
with db_con.connect() as db_conn:
    for query in table_queries:
        db_conn.execute(sqla.text(query))
    print('Tables created')

Tables created


#### Loading cleaned data to created tables

In [6]:
category_translation.to_sql('category_translation',db_con,if_exists='append',index=False)
geolocation.to_sql('geolocation',db_con,if_exists='append',index=False)
customers.to_sql('customers',db_con,if_exists='append',index=False)
sellers.to_sql('sellers',db_con,if_exists='append',index=False)
products.to_sql('products',db_con,if_exists='append',index=False)
orders.to_sql('orders',db_con,if_exists='append',index=False)
order_items.to_sql('order_items',db_con,if_exists='append',index=False)
order_payments.to_sql('order_payments',db_con,if_exists='append',index=False)
order_reviews.to_sql('order_reviews',db_con,if_exists='append',index=False)

print('tables loaded')

tables loaded


In [7]:
# to verify all the data is loaded to sql correctly both sql and data in dataframes data will be compared
tables = {'category_translation': category_translation, 'geolocation': geolocation,'customers': customers, 'sellers': sellers,
        'products': products,'orders': orders, 'order_items': order_items,'order_payments': order_payments, 'order_reviews': order_reviews}

for name in tables:
    # sql query to get the count of the table
    sql_count = pd.read_sql("select count(*) as cnt from " + name, db_con)['cnt'][0]
    print(name, len(tables[name]), sql_count)

category_translation 74 74
geolocation 19015 19015
customers 99441 99441
sellers 3095 3095
products 32951 32951
orders 99441 99441
order_items 112650 112650
order_payments 103877 103877
order_reviews 99224 99224
